# IPD Tournaments and Evolutionary Dynamics

*Week 3 companion notebook — Konstanz 2026, Computational Methods for Modeling Dynamic Social Behavior.*

Last week we clustered **human** PGG time series with DTW and found stable behavioural types (Hausladen, Engel & Schubert, 2026). This week we shift from observation to **mechanism**: given a population of strategies, which ones survive?

We build two things:

1. **An Axelrod-style round-robin tournament** (Axelrod, 1984) — every strategy plays every other strategy in a repeated Prisoner's Dilemma, and we rank them by total score.
2. **A Moran process** — an evolutionary dynamic that lets the population composition change over time, with fitter strategies reproducing more often.

We then add two LLM-inspired strategies motivated by Willis et al. (2025), who had frontier LLMs generate IPD strategies as Python code, and ask: do the LLM strategies survive evolution?

### How this connects to the course

- **Week 2**: We discovered that human participants form persistent behavioural *types* — free riders, conditional cooperators, switchers. Types coexist.
- **Week 3** (this notebook): We discover that strategy *populations* evolve — and typically one type dominates, unless institutional mechanisms (Ostrom, 1990) prevent it.
- **Transfer question**: When LLMs play these games, do they look more like the human pattern (stable coexistence) or the evolutionary pattern (fixation)?

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations

np.random.seed(42)
plt.rcParams['figure.dpi'] = 110

---

## Part 1 — Build an Axelrod Tournament

### The Iterated Prisoner's Dilemma

Two players meet repeatedly. Each round, both simultaneously choose **C** (cooperate) or **D** (defect). The payoffs follow the standard Axelrod values:

| | Opponent C | Opponent D |
|---|---|---|
| **I play C** | R=3, R=3 | S=0, T=5 |
| **I play D** | T=5, S=0 | P=1, P=1 |

The ordering **T > R > P > S** and **2R > T + S** ensures that mutual cooperation is collectively optimal but individual defection is always tempting. Over many rounds, the question becomes: which *strategy* — which rule for choosing C or D based on the history so far — earns the highest cumulative score?

### 1.1 Payoff matrix and match engine

We define the payoff matrix and a function that plays two strategies against each other for a fixed number of rounds.

In [ ]:
# Payoff matrix: (my_payoff, opponent_payoff) indexed by (my_move, opp_move)
PAYOFFS = {
    ('C', 'C'): (3, 3),  # R, R — mutual cooperation
    ('C', 'D'): (0, 5),  # S, T — sucker / temptation
    ('D', 'C'): (5, 0),  # T, S — temptation / sucker
    ('D', 'D'): (1, 1),  # P, P — mutual defection
}


def play_match(strategy_a, strategy_b, rounds=50):
    """
    Play two strategies against each other for `rounds` rounds.
    
    Each strategy is a callable: strategy(my_history, opponent_history) -> 'C' or 'D'
    where my_history and opponent_history are lists of past moves.
    
    Returns (score_a, score_b) — cumulative payoffs.
    """
    history_a, history_b = [], []
    score_a, score_b = 0, 0
    
    for _ in range(rounds):
        move_a = strategy_a(history_a, history_b)
        move_b = strategy_b(history_b, history_a)
        
        payoff_a, payoff_b = PAYOFFS[(move_a, move_b)]
        score_a += payoff_a
        score_b += payoff_b
        
        history_a.append(move_a)
        history_b.append(move_b)
    
    return score_a, score_b

### 1.2 Six classic strategies

Each strategy is a function with the signature `(my_history, opponent_history) -> 'C' or 'D'`. These are the building blocks of Axelrod's original tournaments.

In [ ]:
def always_cooperate(my_history, opp_history):
    """Always cooperate, no matter what."""
    return 'C'


def always_defect(my_history, opp_history):
    """Always defect, no matter what."""
    return 'D'


def tit_for_tat(my_history, opp_history):
    """Cooperate on the first move, then copy opponent's last move."""
    if not opp_history:
        return 'C'
    return opp_history[-1]


def grim_trigger(my_history, opp_history):
    """Cooperate until opponent defects once, then defect forever."""
    if 'D' in opp_history:
        return 'D'
    return 'C'


def random_strategy(my_history, opp_history):
    """Cooperate or defect with equal probability."""
    return np.random.choice(['C', 'D'])


def pavlov(my_history, opp_history):
    """Win-stay, lose-shift. Cooperate if both chose the same last round, else defect.
    Cooperate on the first move."""
    if not my_history:
        return 'C'
    # "Win" = both same (CC or DD), "Lose" = different (CD or DC)
    if my_history[-1] == opp_history[-1]:
        return 'C'
    else:
        return 'D'


# Collect strategies in a dict for easy iteration
STRATEGIES = {
    'Always C': always_cooperate,
    'Always D': always_defect,
    'Tit-for-Tat': tit_for_tat,
    'Grim Trigger': grim_trigger,
    'Random': random_strategy,
    'Pavlov': pavlov,
}

print(f'Loaded {len(STRATEGIES)} strategies: {", ".join(STRATEGIES.keys())}')

### 1.3 Round-robin tournament

Every strategy plays every other strategy (including itself) in a 50-round match. We store the score each strategy earns against each opponent.

In [ ]:
def run_tournament(strategies, rounds=50, seed=42):
    """
    Round-robin tournament. Returns a DataFrame of scores:
    rows = player strategy, columns = opponent strategy, values = player's score.
    """
    np.random.seed(seed)
    names = list(strategies.keys())
    n = len(names)
    scores = pd.DataFrame(0, index=names, columns=names, dtype=float)
    
    for i in range(n):
        for j in range(n):
            sa, sb = play_match(strategies[names[i]], strategies[names[j]], rounds=rounds)
            scores.iloc[i, j] = sa
    
    return scores


scores = run_tournament(STRATEGIES)
scores

### 1.4 Visualize the tournament

Two views: a **heatmap** showing pairwise scores and a **bar chart** of total scores across all opponents.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True,
                         gridspec_kw={'width_ratios': [1.3, 1]})

# Heatmap of pairwise scores
sns.heatmap(scores, annot=True, fmt='.0f', cmap='YlOrRd', ax=axes[0],
            cbar_kws={'label': 'score'})
axes[0].set_title('Pairwise scores (row = player, col = opponent)')
axes[0].set_ylabel('player'); axes[0].set_xlabel('opponent')

# Bar chart of total scores
totals = scores.sum(axis=1).sort_values(ascending=True)
colors = ['tab:green' if n == 'Tit-for-Tat' else 'tab:blue' for n in totals.index]
axes[1].barh(totals.index, totals.values, color=colors)
axes[1].set_xlabel('total score (sum across all opponents)')
axes[1].set_title('Tournament ranking')

plt.show()

### Reading the results

The heatmap tells a story Axelrod (1984) made famous:

- **Always Defect** beats every individual opponent (look at any column — its row is never the lowest), yet it finishes near the bottom of the total ranking. Why? Because it only ever earns P=1 against itself and other defectors, while cooperative strategies earn R=3 against each other.
- **Tit-for-Tat** never *beats* any opponent in a head-to-head match — it only ties or loses the first-round advantage — yet it accumulates the highest total. It cooperates with cooperators (earning R=3 every round) and retaliates against defectors (limiting losses).
- **Always Cooperate** is the most exploitable: it hands 5 points per round to Always Defect.

The lesson: in a *tournament* (where you face a diverse field), being exploitative is less important than being *consistently good against everyone*.

### 1.5 Axelrod's four properties

Axelrod (1984) identified four properties that distinguished successful strategies in his tournaments:

1. **Nice** — never defects first (cooperates on round 1 and never defects unprovoked)
2. **Retaliatory** — punishes defection (defects after opponent defects)
3. **Forgiving** — returns to cooperation after retaliating if the opponent returns to C
4. **Clear** — deterministic, so the opponent can learn to predict and cooperate with it

We test each strategy against these criteria programmatically.

In [ ]:
def axelrod_properties(strategy):
    """
    Test a strategy for Axelrod's four properties.
    Returns a dict with boolean values for: nice, retaliatory, forgiving, clear.
    """
    props = {}
    
    # --- Nice: cooperates on round 1, and never defects first against AllC ---
    first_move = strategy([], [])
    # Play 20 rounds against AllC and check if strategy ever defects
    my_h, opp_h = [], []
    defects_against_allc = False
    for _ in range(20):
        m = strategy(my_h, opp_h)
        if m == 'D':
            defects_against_allc = True
            break
        my_h.append(m)
        opp_h.append('C')
    props['nice'] = (first_move == 'C') and not defects_against_allc
    
    # --- Retaliatory: defects after opponent defects ---
    # Simulate: cooperate for 3 rounds, then opponent defects on round 4
    my_h = ['C', 'C', 'C']
    opp_h = ['C', 'C', 'D']  # opponent defected on round 3
    response = strategy(my_h, opp_h)
    props['retaliatory'] = (response == 'D')
    
    # --- Forgiving: returns to C after retaliating, if opponent returns to C ---
    # Simulate: opponent defected once, strategy retaliated, then opponent cooperates
    my_h = ['C', 'C', 'C', 'D']   # I retaliated on round 4
    opp_h = ['C', 'C', 'D', 'C']  # opponent cooperated on round 4
    # Check if strategy returns to C within the next 3 rounds of mutual cooperation
    forgives = False
    for _ in range(3):
        m = strategy(my_h, opp_h)
        if m == 'C':
            forgives = True
            break
        my_h.append(m)
        opp_h.append('C')
    props['forgiving'] = forgives
    
    # --- Clear: deterministic (run the same scenario twice, get the same answer) ---
    # Test with multiple histories and check for consistency across 10 trials
    test_scenarios = [
        ([], []),
        (['C'], ['C']),
        (['C'], ['D']),
        (['C', 'D'], ['D', 'C']),
    ]
    deterministic = True
    for my_h_test, opp_h_test in test_scenarios:
        results = set()
        for _ in range(20):
            results.add(strategy(list(my_h_test), list(opp_h_test)))
        if len(results) > 1:
            deterministic = False
            break
    props['clear'] = deterministic
    
    return props


# Build and display the properties table
props_data = {name: axelrod_properties(fn) for name, fn in STRATEGIES.items()}
props_df = pd.DataFrame(props_data).T
# Display with checkmarks
display_df = props_df.replace({True: '\u2713', False: '\u2717'})
display_df

### The properties tell the tournament story

Axelrod found that successful strategies tend to be **nice**, **retaliatory**, **forgiving**, and **clear**:

- **Tit-for-Tat** has all four properties. It cooperates first (nice), punishes defection immediately (retaliatory), forgives as soon as the opponent cooperates (forgiving), and is fully deterministic (clear). This is why it wins.
- **Grim Trigger** is nice, retaliatory, and clear — but it **lacks forgiveness**. A single defection triggers permanent punishment. In a noisy world (or a world with Random players), this is costly.
- **Always Defect** lacks niceness. It never gives cooperation a chance, so it never earns the mutual-cooperation payoff R=3.
- **Random** lacks clarity. Its opponent cannot learn to cooperate with it, because its behavior is unpredictable.
- **Pavlov** is nice, forgiving, and clear, but it is not retaliatory in the standard test: after a single opponent defection, Pavlov switches to D (because the moves differed), which *looks* retaliatory — but in some histories it cooperates after being exploited. Pavlov's logic is subtler: it is a *self-correcting* strategy.

---

## Part 2 — Add Evolution: The Moran Process

A round-robin tournament tells us which strategy earns the most *in a fixed field*. But what if the field itself evolves? If fitter strategies reproduce and less-fit ones die out, the population composition changes — and with it the environment each strategy faces.

The **Moran process** is a simple model of this:

1. Start with a population of agents, each playing a fixed strategy.
2. Each generation: compute each agent's **fitness** (average score in IPD matches against all other agents).
3. Select one agent to **reproduce** with probability proportional to fitness.
4. Select one agent **uniformly at random** to be replaced by the offspring (same strategy as the parent).
5. Repeat.

Over time, fitter strategies spread. Eventually the population may reach **fixation** — a state where all agents play the same strategy.

In [ ]:
def moran_process(strategies, pop_size=60, generations=500, rounds_per_match=50, seed=42):
    """
    Run a Moran process on a population of IPD strategies.
    
    Parameters
    ----------
    strategies : dict
        {name: callable} for each strategy type.
    pop_size : int
        Total population size (divided equally among strategy types).
    generations : int
        Number of generations to simulate.
    rounds_per_match : int
        Rounds per IPD match.
    seed : int
        Random seed.
    
    Returns
    -------
    history : pd.DataFrame
        Columns = strategy names, rows = generation, values = count in population.
    """
    rng = np.random.default_rng(seed)
    names = list(strategies.keys())
    n_types = len(names)
    
    # Initialize: equal numbers of each type
    agents_per_type = pop_size // n_types
    population = []  # list of strategy names
    for name in names:
        population.extend([name] * agents_per_type)
    # Fill remaining slots with random types
    while len(population) < pop_size:
        population.append(rng.choice(names))
    population = list(population)
    
    # Track composition
    history = []
    
    for gen in range(generations):
        # Record current composition
        counts = {name: population.count(name) for name in names}
        history.append(counts)
        
        # Compute fitness: average score vs. all other agents
        # For efficiency, compute pairwise scores between types (not individuals)
        # and weight by the number of opponents of each type
        type_vs_type = {}
        for a in names:
            for b in names:
                # Use deterministic seed per pair per generation for reproducibility
                old_state = np.random.get_state()
                np.random.seed(seed + gen * 1000 + names.index(a) * 100 + names.index(b))
                sa, _ = play_match(strategies[a], strategies[b], rounds=rounds_per_match)
                np.random.set_state(old_state)
                type_vs_type[(a, b)] = sa
        
        # Each agent's fitness = weighted average score vs population
        fitness = np.zeros(pop_size)
        for i, agent_type in enumerate(population):
            total = 0.0
            n_opponents = 0
            for opp_type in names:
                # Number of opponents of this type (subtract 1 if same type, for self)
                n_opp = counts[opp_type]
                if opp_type == agent_type:
                    n_opp -= 1
                if n_opp > 0:
                    total += type_vs_type[(agent_type, opp_type)] * n_opp
                    n_opponents += n_opp
            fitness[i] = total / max(n_opponents, 1)
        
        # Ensure non-negative fitness for selection
        fitness = fitness - fitness.min() + 1e-6
        
        # Select parent proportional to fitness
        probs = fitness / fitness.sum()
        parent_idx = rng.choice(pop_size, p=probs)
        
        # Select agent to replace uniformly at random
        replace_idx = rng.integers(pop_size)
        
        # Replace
        population[replace_idx] = population[parent_idx]
    
    # Record final state
    history.append({name: population.count(name) for name in names})
    
    return pd.DataFrame(history)

### 2.1 Run the Moran process

We run the process three times with different seeds to see stochastic variation. The Moran process is inherently noisy — each run can end differently.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True, sharey=True)

# Color palette for strategies
palette = {
    'Always C': '#2ca02c',
    'Always D': '#d62728',
    'Tit-for-Tat': '#1f77b4',
    'Grim Trigger': '#ff7f0e',
    'Random': '#9467bd',
    'Pavlov': '#8c564b',
}

seeds = [42, 123, 7]
for ax, s in zip(axes, seeds):
    history = moran_process(STRATEGIES, pop_size=60, generations=500,
                            rounds_per_match=50, seed=s)
    for name in STRATEGIES:
        ax.plot(history.index, history[name], label=name, color=palette[name], lw=1.5)
    ax.set_xlabel('generation')
    ax.set_title(f'seed = {s}')
    ax.set_ylim(0, 62)

axes[0].set_ylabel('population count')
axes[2].legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
fig.suptitle('Moran process: 6 strategies, 60 agents, 500 generations', fontsize=12)
plt.show()

### What is fixation?

**Fixation** occurs when one strategy takes over the entire population — all other types go extinct. In the Moran process, fixation is the only absorbing state (without mutation). The question is not *whether* it happens, but *which* strategy fixes and *how long* it takes.

Notice that the three runs may produce different outcomes. The Moran process is stochastic: even a strategy with lower average fitness can occasionally fix, because the random replacement step can remove fit agents by chance. This is **genetic drift**, and it matters more in small populations.

### Round-robin ranking vs. evolutionary success

**Which strategies survive? Is it always the same ones that won the round-robin? Why might the ranking differ?**

In the round-robin, every strategy faces a fixed, uniform field. In evolution, the field *changes*: as Always Cooperate gets exploited and declines, Always Defect loses its easy prey and becomes less fit. Strategies that thrive in a *changing* ecology may differ from those that thrive in a static one.

### Connection to Willis et al. (2025)

This is exactly the framework Willis et al. use — but instead of hand-coded strategies like ours, they plug in **LLM-generated** ones. They prompted GPT-4o, Claude, and other frontier models to write Python functions with the same `(my_history, opp_history) -> 'C'/'D'` signature, then ran Axelrod tournaments and evolutionary dynamics. The key finding: GPT-4o-generated strategies evolved toward aggression, while Claude-generated strategies evolved toward cooperation. We will test simplified versions of those strategies in Part 3.

---

## Part 3 — The Transfer Question: LLM-Inspired Strategies

Willis et al. (2025) found that different LLMs generate strategies with distinctive "personalities":

- **GPT-4o** strategies tend to be aggressive: they test opponents early, then exploit any sign of weakness and never forgive.
- **Claude** strategies tend to be cooperative: they use tit-for-tat-like logic but with built-in forgiveness.

We code simplified versions of these findings as two new strategies.

In [ ]:
def llm_aggressive(my_history, opp_history):
    """LLM-aggressive (mimics GPT-4o finding from Willis et al.):
    Cooperate for the first 3 rounds to 'test' the opponent.
    If opponent ever defected, defect forever (never forgive)."""
    if len(my_history) < 3:
        return 'C'
    if 'D' in opp_history:
        return 'D'
    return 'C'


def llm_cooperative(my_history, opp_history):
    """LLM-cooperative (mimics Claude finding from Willis et al.):
    Tit-for-tat with forgiveness — if opponent defects, retaliate once,
    then try cooperating again."""
    if not opp_history:
        return 'C'
    if opp_history[-1] == 'D':
        # Retaliate
        return 'D'
    # If I just retaliated (I played D) but opponent is now cooperating, forgive
    if len(my_history) >= 2 and my_history[-1] == 'D' and opp_history[-1] == 'C':
        return 'C'
    return 'C'


# Extended strategy set
STRATEGIES_EXT = {**STRATEGIES,
                  'LLM-Aggressive': llm_aggressive,
                  'LLM-Cooperative': llm_cooperative}

print(f'Extended tournament: {len(STRATEGIES_EXT)} strategies')

### 3.1 Re-run the round-robin with LLM strategies

In [ ]:
scores_ext = run_tournament(STRATEGIES_EXT)

fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True,
                         gridspec_kw={'width_ratios': [1.5, 1]})

sns.heatmap(scores_ext, annot=True, fmt='.0f', cmap='YlOrRd', ax=axes[0],
            cbar_kws={'label': 'score'})
axes[0].set_title('Pairwise scores (extended tournament)')
axes[0].set_ylabel('player'); axes[0].set_xlabel('opponent')

totals_ext = scores_ext.sum(axis=1).sort_values(ascending=True)
colors_ext = ['tab:orange' if 'LLM' in n else 'tab:blue' for n in totals_ext.index]
axes[1].barh(totals_ext.index, totals_ext.values, color=colors_ext)
axes[1].set_xlabel('total score')
axes[1].set_title('Tournament ranking (orange = LLM-inspired)')

plt.show()

### 3.2 Re-run the Moran process with LLM strategies

In [ ]:
palette_ext = {**palette,
               'LLM-Aggressive': '#e377c2',
               'LLM-Cooperative': '#17becf'}

fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True, sharey=True)

seeds = [42, 123, 7]
for ax, s in zip(axes, seeds):
    history = moran_process(STRATEGIES_EXT, pop_size=64, generations=500,
                            rounds_per_match=50, seed=s)
    for name in STRATEGIES_EXT:
        ax.plot(history.index, history[name], label=name,
                color=palette_ext[name], lw=1.5)
    ax.set_xlabel('generation')
    ax.set_title(f'seed = {s}')
    ax.set_ylim(0, 66)

axes[0].set_ylabel('population count')
axes[2].legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
fig.suptitle('Moran process with LLM-inspired strategies (8 types, 64 agents)', fontsize=12)
plt.show()

### 3.3 Axelrod properties of the LLM strategies

In [ ]:
llm_props = {
    'LLM-Aggressive': axelrod_properties(llm_aggressive),
    'LLM-Cooperative': axelrod_properties(llm_cooperative),
}
llm_props_df = pd.DataFrame(llm_props).T
# Combine with original for comparison
all_props = pd.concat([props_df, llm_props_df])
all_props.replace({True: '\u2713', False: '\u2717'})

### What the properties predict

Willis et al. (2025) found that GPT-4o strategies evolve toward aggression and Claude strategies toward cooperation. Our simplified versions show consistent dynamics:

- **LLM-Aggressive** is nice (it cooperates first) and retaliatory, but it **lacks forgiveness** — just like Grim Trigger. Axelrod's framework predicts it will struggle in diverse populations because it cannot recover from accidental or exploratory defections.
- **LLM-Cooperative** has all four properties: nice, retaliatory, forgiving, and clear. Like Tit-for-Tat, it cooperates when it can and punishes when it must — but it always leaves the door open for reconciliation.

The lesson maps onto a broader theme in the LLM literature: models that generate *forgiving* strategies produce populations that sustain cooperation; models that generate *unforgiving* strategies produce populations that collapse into mutual defection.

### Exercise

**For your project:** What strategy would *your* chosen LLM generate? Code it as a function with the same `(my_history, opp_history) -> 'C'/'D'` signature and drop it into this tournament. Run the round-robin and the Moran process. Does it survive? What Axelrod properties does it have?

---

## Part 4 — Connection to Patterned Heterogeneity

Look at the Moran process plots again. The population typically evolves through recognizable **phases**:

1. **Early diversity** — all strategy types coexist at roughly equal frequency.
2. **Competition** — weaker strategies decline as stronger ones grow. The population becomes less diverse.
3. **Fixation or coexistence** — either one strategy takes over completely, or a small number of strategies settle into a dynamic equilibrium.

### Connecting to Week 2

In Week 2, we clustered human PGG participants and found stable behavioural types — free riders, conditional cooperators, switchers (Hausladen, Engel & Schubert, 2026). Those types **coexist persistently** across all 10 rounds: free riders do not drive out cooperators, and cooperators do not convert free riders.

In evolutionary dynamics, the pattern is different: one type often **dominates** — unless institutional mechanisms prevent it. This is precisely Ostrom's (1990) insight: real-world commons persist not because defectors are absent, but because **institutions** (monitoring, graduated sanctions, conflict resolution) keep the ecology of strategies in balance.

The contrast:
- **Human experiments**: types coexist because the game has finite rounds and no reproduction. The ecology is frozen.
- **Evolutionary simulations**: types compete because reproduction amplifies fitness differences. The ecology is fluid.
- **Real-world commons** (Ostrom): types coexist *and* compete, but institutional rules constrain the dynamics — creating stable, diverse communities.

When we study LLMs playing repeated games, which pattern do they follow? Willis et al. (2025) suggest the evolutionary one — strategies evolve toward fixation. But the human data says coexistence is the norm. One of the big open questions for this course: can we design LLM interactions that sustain the *human* pattern?

### Final exercises

**Exercise 1: Graduated sanctions.** Modify `llm_cooperative` to add Ostrom's principle 5 — graduated sanctions. Instead of always retaliating with a single defection, punish *harder* on repeated defection: if the opponent has defected in 2 of the last 3 rounds, defect for 2 rounds; if 3 of the last 3, defect for 3 rounds. Does it survive better in the Moran process?

```python
def llm_cooperative_graduated(my_history, opp_history):
    """Tit-for-tat with graduated sanctions."""
    if not opp_history:
        return 'C'
    # Count opponent defections in the last 3 rounds
    recent = opp_history[-3:] if len(opp_history) >= 3 else opp_history
    n_defections = recent.count('D')
    # Graduated punishment: punish for as many rounds as recent defections
    if n_defections > 0:
        # Check if I have already punished enough
        recent_mine = my_history[-n_defections:] if len(my_history) >= n_defections else my_history
        punishment_served = recent_mine.count('D')
        if punishment_served < n_defections:
            return 'D'
    return 'C'
```

**Exercise 2: The switcher strategy.** Can you design a "switcher" strategy — one that alternates between cooperation and defection on a regular cycle? How does it perform in the tournament vs. in evolution? Is it exploitable?

```python
def switcher(my_history, opp_history):
    """Alternate between C and D every round."""
    if len(my_history) % 2 == 0:
        return 'C'
    else:
        return 'D'
```

---

## Suggested further reading

- **Axelrod, R. (1984).** *The Evolution of Cooperation.* Basic Books. — The original tournament framework and the four properties of successful strategies.
- **Nowak, M. A. (2006).** *Evolutionary Dynamics: Exploring the Equations of Life.* Harvard University Press. — The Moran process, fixation probabilities, and the mathematics of evolutionary game theory.
- **Willis, J. et al. (2025).** *LLM-Generated IPD Strategies.* — LLMs generating IPD strategies as Python code; evolutionary tournaments reveal model-specific "personalities."
- **Ostrom, E. (1990).** *Governing the Commons.* Cambridge University Press. — How real-world communities sustain cooperation through institutional design.
- **Hausladen, C. I., Engel, C. & Schubert, R. (2026).** *Identifying Latent Intentions via IRL in Repeated Linear Public Good Games.* — The PGG type-discovery pipeline from Week 2.